# Демо: магические методы и `@dataclass`

Прокликай Shift+Enter каждую ячейку и посмотри, как Python автоматически зовёт ваши методы при встроенных операциях (`len()`, `==`, `print`, `obj[i]`, `obj()`), и как `@dataclass` генерирует шаблонный код за вас. В конце — три мини-задания.

## Часть 1. `__repr__` и `__str__` — строковое представление

Сейчас посмотрим, что показывает Python, когда вы пишете `print(obj)`. По умолчанию — скучное `<__main__.Point object at 0x7f...>`. С `__repr__` объект сам говорит, кто он:

In [1]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

p = Point(3, 4)
print(p)              # <__main__.Point object at 0x...>
print(repr(p))        # то же самое — нет ни __repr__, ни __str__

In [2]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Point(x={self.x}, y={self.y})"

p = Point(3, 4)
print(p)              # Point(x=3, y=4)
print(repr(p))        # Point(x=3, y=4)
print([p, p])         # [Point(x=3, y=4), Point(x=3, y=4)] — repr виден в списках
print(f"тут точка {p}")  # тут точка Point(x=3, y=4)

Point(x=3, y=4)
Point(x=3, y=4)
[Point(x=3, y=4), Point(x=3, y=4)]
тут точка Point(x=3, y=4)


Если хочется разделять — однозначное представление для дебага и человекочитаемое для пользователя — определяют **оба** метода. `print(obj)` вызовет `__str__`, а `repr(obj)` — `__repr__`.

In [3]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Point(x={self.x}, y={self.y})"

    def __str__(self):
        return f"точка ({self.x}, {self.y})"

p = Point(3, 4)
print(p)              # точка (3, 4) — __str__
print(repr(p))        # Point(x=3, y=4) — __repr__
print([p, p])         # [Point(x=3, y=4), ...] — внутри list зовётся repr

точка (3, 4)
Point(x=3, y=4)
[Point(x=3, y=4), Point(x=3, y=4)]


## Часть 2. `__eq__` и `__hash__` — равенство и хэш

По умолчанию Python сравнивает объекты по идентичности (адресу в памяти): два разных `Point(3, 4)` будут не равны. Часто хочется сравнивать по содержимому — реализуем `__eq__`:

In [4]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return self.x == other.x and self.y == other.y

print(Point(3, 4) == Point(3, 4))   # True — равны по содержимому
print(Point(3, 4) == Point(5, 6))   # False
print(Point(3, 4) == "строка")     # False — NotImplemented сработал, Python попробовал обратное

True
False
False


А что если попробовать сложить такие точки в `set` или сделать ключом `dict`? Получим неприятный сюрприз — Python не пускает:

In [5]:
try:
    s = {Point(3, 4), Point(3, 4)}
except TypeError as e:
    print(f"TypeError: {e}")

TypeError: unhashable type: 'Point'


Когда вы определили `__eq__`, Python автоматически делает класс **нехэшируемым** (`__hash__` = `None`). Контракт жёсткий: равные объекты обязаны иметь равные хэши. Чтобы починить — определяем `__hash__` через те же атрибуты:

In [6]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return self.x == other.x and self.y == other.y

    def __hash__(self):
        return hash((self.x, self.y))   # стандартный приём — хэш кортежа атрибутов

s = {Point(3, 4), Point(3, 4), Point(5, 6)}
print(len(s))         # 2 — две точки совпали и слились в одну

d = {Point(0, 0): "начало координат"}
print(d[Point(0, 0)])  # начало координат — ключ нашёлся по содержимому

2
начало координат


## Часть 3. `__len__` и `__getitem__` — sequence-протокол

Реализуем `__len__` — чтобы работал `len(obj)`. Бонус: объект с `__len__` автоматически становится truthy/falsy по правилу «пустое — ложь, непустое — правда».

In [7]:
class Playlist:
    def __init__(self, tracks):
        self.tracks = tracks

    def __len__(self):
        return len(self.tracks)

p = Playlist(["song1", "song2", "song3"])
print(len(p))          # 3

if p:                  # __len__ → 3 → truthy
    print("есть треки")

empty = Playlist([])
if not empty:           # __len__ → 0 → falsy
    print("пусто")

3
есть треки
пусто


Добавляем `__getitem__` — теперь работает индексация `obj[i]`. Если делегируем всё внутреннему списку, бесплатно работают и срезы (`p[1:3]`). А пара `__len__` + `__getitem__` (с целым индексом от 0) даёт **sequence-протокол** — Python автоматически разрешает `for`-цикл и `list(obj)`:

In [8]:
class Playlist:
    def __init__(self, tracks):
        self.tracks = tracks

    def __len__(self):
        return len(self.tracks)

    def __getitem__(self, index):
        return self.tracks[index]

p = Playlist(["song1", "song2", "song3"])
print(p[0])            # song1
print(p[-1])            # song3 — отрицательная индексация работает
print(p[0:2])           # ['song1', 'song2'] — срезы тоже

# for-цикл работает БЕЗ __iter__ — Python подхватывает sequence-протокол
for track in p:
    print("  iter:", track)

print(list(p))          # ['song1', 'song2', 'song3']
import random
print(random.choice(p)) # один из треков

song1
song3
['song1', 'song2']
  iter: song1
  iter: song2
  iter: song3
['song1', 'song2', 'song3']
song1


## Часть 4. `__add__` — оператор `+` для своих объектов

Когда Python видит `a + b`, он зовёт `a.__add__(b)`. Реализовав `__add__`, можно учить свои объекты складываться:

In [9]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return Vector(self.x + other.x, self.y + other.y)

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

v1 = Vector(1, 2)
v2 = Vector(3, 4)
print(v1 + v2)        # Vector(4, 6)

Vector(4, 6)


## Часть 5. `__call__` — объект как функция

Если у класса определён `__call__`, объекты можно вызывать как функции через скобки. По сути — настраиваемая функция через объект, альтернатива замыканиям из прошлой недели.

In [10]:
class Adder:
    def __init__(self, base):
        self.base = base

    def __call__(self, x):
        return x + self.base

add5 = Adder(5)        # создали объект
print(callable(add5))  # True — у объекта есть __call__

# Объект «работает» как функция
print(add5(10))        # 15
print(add5(100))       # 105

add100 = Adder(100)
print(add100(7))       # 107 — у каждого свой base

True
15
105
107


## Часть 6. `@dataclass` — автогенерация шаблонного кода

Посмотрим на класс `Point` без декоратора и с ним. Декоратор `@dataclass` читает аннотации типов и автогенерирует `__init__`, `__repr__`, `__eq__`. Кода становится в разы меньше.

In [11]:
from dataclasses import dataclass

# Старый вариант — три метода руками:
class PointManual:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __repr__(self):
        return f"PointManual(x={self.x}, y={self.y})"
    def __eq__(self, other):
        if not isinstance(other, PointManual):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

# Новый вариант — три строки:
@dataclass
class Point:
    x: int
    y: int

p = Point(3, 4)
print(p)                       # Point(x=3, y=4) — авто __repr__
print(Point(3, 4) == Point(3, 4))  # True — авто __eq__
print(p.x, p.y)                # 3 4 — авто __init__

Point(x=3, y=4)
True
3 4


Атрибуты можно задавать значения по умолчанию. Параметры без дефолта должны идти **до** параметров с дефолтом — точно как в обычных функциях:

In [12]:
@dataclass
class User:
    name: str
    age: int
    is_active: bool = True

alice = User("Аня", 25)
print(alice)                  # User(name='Аня', age=25, is_active=True)

bob = User("Боря", 30, is_active=False)
print(bob)                    # User(name='Боря', age=30, is_active=False)

User(name='Аня', age=25, is_active=True)
User(name='Боря', age=30, is_active=False)


## Часть 7. Изменяемый дефолт через `field(default_factory=...)`

На прошлой неделе мы видели mutable default trap в обычных функциях. `@dataclass` эту ловушку явно блокирует — Python падает при попытке поставить `[]` в дефолт:

In [13]:
from dataclasses import dataclass

try:
    @dataclass
    class Bad:
        items: list = []     # ValueError
except ValueError as e:
    print(f"ValueError: {e}")

ValueError: mutable default <class 'list'> for field items is not allowed: use default_factory


Правильный способ — `field(default_factory=list)`. Каждый экземпляр получит свой свежий `list()`, ловушки нет:

In [14]:
from dataclasses import dataclass, field

@dataclass
class User:
    name: str
    tags: list = field(default_factory=list)

alice = User("Аня")
bob = User("Боря")

alice.tags.append("admin")
print(alice.tags)            # ['admin']
print(bob.tags)              # [] — у Бори свой пустой список, всё хорошо

['admin']
[]


## Часть 8. `@dataclass(frozen=True)` — неизменяемый класс

Если объект не должен меняться после создания (типичный пример — конфиг эксперимента), передаём `frozen=True`. Бонус: автоматически генерируется `__hash__`, объект годится как ключ `dict` или элемент `set`:

In [15]:
from dataclasses import dataclass

@dataclass(frozen=True)
class ModelConfig:
    learning_rate: float
    batch_size: int
    epochs: int

cfg = ModelConfig(learning_rate=0.001, batch_size=32, epochs=10)
print(cfg)                  # ModelConfig(...)

# Запрет на модификацию
try:
    cfg.learning_rate = 0.01
except Exception as e:
    print(f"{type(e).__name__}: {e}")

# Хэшируемость — можно как ключ dict
ratings = {cfg: 0.95}
print(ratings)
print(ratings[ModelConfig(0.001, 32, 10)])   # 0.95 — нашли по содержимому

ModelConfig(learning_rate=0.001, batch_size=32, epochs=10)
FrozenInstanceError: cannot assign to field 'learning_rate'
{ModelConfig(learning_rate=0.001, batch_size=32, epochs=10): 0.95}
0.95


## Мини-задания

Три коротких упражнения. Подсказок к именам и методам нет — вспомни сам.

**Задание 1.** Напиши класс `Money` с полями `amount: int` и `currency: str`. Реализуй `__repr__`, `__eq__` (равенство только при совпадении и суммы, и валюты), `__add__` (складывает только одинаковые валюты — иначе `TypeError`).

**Задание 2.** Перепиши `Money` через `@dataclass(frozen=True)`. Покажи, что объект годится как ключ словаря.

**Задание 3.** Что напечатает код ниже? Сначала угадай, потом запусти.

In [16]:
# Задание 1
# class Money:
#     ...

# Проверка:
# usd_5 = Money(5, "USD")
# print(usd_5)                       # Money(amount=5, currency='USD')
# print(usd_5 == Money(5, "USD"))    # True
# print(usd_5 + Money(10, "USD"))    # Money(amount=15, currency='USD')
# usd_5 + Money(7, "EUR")            # TypeError


In [17]:
# Задание 2
# from dataclasses import dataclass
#
# @dataclass(frozen=True)
# class Money:
#     ...

# Проверка:
# prices = {Money(100, "USD"): "laptop", Money(50, "USD"): "keyboard"}
# print(prices[Money(100, "USD")])  # laptop


In [18]:
# Задание 3 — твой прогноз для каждой строки впиши в комментарий:
from dataclasses import dataclass

@dataclass
class Box:
    label: str
    items: list

shared = []
b1 = Box("red", shared)
b2 = Box("blue", shared)   # передали тот же список!
b1.items.append("apple")

print(b1.items)            # ?
print(b2.items)            # ?
print(b1 == b2)            # ?


['apple']
['apple']
False
